# 🤖 Workshop: Agentes de IA e MCP (Model Context Protocol)

Neste notebook você vai aprender:
1. O que são agentes de IA
2. Como conectar agentes via MCP (protocolo)
3. Como usar servidores MCP via stdio
4. Como criar seu próprio servidor MCP

**100% Gratuito**: Gemini + APIs públicas

## 📦 Instalação

In [ ]:
!pip install -q langchain_mcp_adapters
!pip install -q langgraph
!pip install -q langchain-google-genai
!pip install -q google-generativeai
!pip install -q mcp

print("✅ Instalação concluída!")

✅ Instalação concluída!


## 🔑 Configuração da API

### Como obter sua chave do Gemini (100% GRÁTIS):

1. Acesse: https://aistudio.google.com/app/apikey
2. Clique em "Create API Key"
3. Copie sua chave
4. Cole abaixo quando solicitado


In [ ]:
from google.colab import userdata
import os

try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    print("✅ Chave carregada")
except:
    from getpass import getpass
    GOOGLE_API_KEY = getpass('Cole sua chave: ')

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

✅ Chave carregada


In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.prompts import PromptTemplate
import asyncio

---
# 🎯 Parte 1: O que é um Agente?

## 🧠 Definição

Um **agente** é uma LLM com superpoderes! Diferenças:

| LLM Normal | Agente |
|------------|--------|
| Apenas responde | Toma decisões |
| Conhecimento limitado | Pode buscar informações atualizadas |
| Resposta única | Múltiplas etapas |
| Sem ferramentas | Usa tools para agir |

### 💡 Exemplo Prático

**Pergunta:** "Qual o clima hoje?"

- **LLM normal:** "Desculpe, meu conhecimento tem uma data de corte"
- **Agente:** *Usa ferramenta de clima* → "Está 28°C e ensolarado em São Paulo!"

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

prompt = PromptTemplate(
    template="""Você é um assistente especializado em livros.
Responda APENAS sobre livros.

Pergunta: {input}""",
    input_variables=["input"]
)

agente_simples = prompt | llm
print("✅ Agente criado!")

✅ Agente criado!


In [ ]:
response = agente_simples.invoke({"input": "Me recomende um livro de ficção"})
print(response.content)

Claro! Recomendo "1984" de George Orwell.

É um clássico da ficção distópica que explora temas como totalitarismo, vigilância e a manipulação da verdade. É uma leitura instigante e muito relevante, mesmo décadas após sua publicação.


In [ ]:
response = agente_simples.invoke({"input": "Qual a previsão do tempo para hoje?"})
print(response.content)

Como um assistente especializado em livros, não tenho informações sobre a previsão do tempo. Posso ajudar com algo relacionado a livros?


---
# 🌐 Parte 2: O que é MCP?

**MCP (Model Context Protocol)** = Protocolo padronizado para conectar LLMs a recursos externos

### Componentes:
- **Server MCP**: Expõe ferramentas
- **Client MCP**: Consome as ferramentas
- **Protocolos**: stdio (local) ou sse (remoto)

### Vantagem:
Crie uma vez, use em qualquer lugar!

## 📡 Entendendo os Protocolos de Transporte

MCP usa diferentes **protocolos de transporte** para comunicação. Vamos entender cada um:

### 🔌 stdio (Standard Input/Output)

**O que é?**
- **stdio** = Comunicação via entrada/saída padrão de processos
- É como dois programas "conversando" através de texto
- Um programa escreve (stdout), o outro lê (stdin)

**Como funciona?**
```
Cliente MCP          Servidor MCP
     |                    |
     |--[mensagem JSON]-->|
     |                    | (processa)
     |<--[resposta JSON]--|
     |                    |
```

**Características:**
- ✅ **Local**: Servidor roda na mesma máquina
- ✅ **Rápido**: Comunicação direta, sem rede
- ✅ **Simples**: Apenas texto via stdin/stdout
- ❌ **Limitado**: Não funciona para servidores remotos

**Exemplo de uso:**
```python
{
    "weather": {
        "command": "python",              # Comando para iniciar
        "args": ["servidor_clima.py"],   # Argumentos
        "transport": "stdio"              # ← Usa stdin/stdout
    }
}
```

**Quando usar stdio?**
- ✅ Desenvolvimento local
- ✅ Ferramentas pessoais
- ✅ Scripts Python/Node.js locais
- ✅ Testes e prototipagem

### 🌐 SSE (Server-Sent Events)

**O que é?**
- **SSE** = Protocolo HTTP para streaming de eventos do servidor
- Conexão HTTP persistente que envia dados em tempo real
- Padrão web (HTML5) para comunicação unidirecional

**Como funciona?**
```
Cliente MCP          Servidor MCP (Internet)
     |                         |
     |--[HTTP GET /sse]------->|
     |                         | (conexão fica aberta)
     |<--[evento: dados]-------|
     |<--[evento: dados]-------|
     |<--[evento: dados]-------|
     |                         |
```

**Características:**
- ✅ **Remoto**: Servidor pode estar em qualquer lugar
- ✅ **Tempo Real**: Recebe atualizações instantâneas
- ✅ **HTTP**: Funciona através de firewalls/proxies
- ✅ **Escalável**: Suporta múltiplos clientes
- ❌ **Mais complexo**: Requer servidor web

**Exemplo de uso:**
```python
{
    "brave_search": {
        "transport": "sse",                        # ← Usa HTTP/SSE
        "url": "https://mcp.bravesearch.com/sse", # URL remota
        "headers": {                               # Headers HTTP opcionais
            "Authorization": "Bearer TOKEN"
        }
    }
}
```

**Quando usar SSE?**
- ✅ Servidores remotos/cloud
- ✅ APIs de terceiros
- ✅ Serviços compartilhados
- ✅ Produção/deploy

**Diferença entre SSE e WebSocket:**
- **SSE**: Unidirecional (servidor → cliente)
- **WebSocket**: Bidirecional (ambos lados)
- Para MCP, SSE é suficiente!

----

# **VAMOS AO VSCODE**

----

---
# 🎓 Resumo

## ✅ Aprendemos:

1. **Agentes**: LLMs que tomam decisões
2. **MCP**: Protocolo padronizado para conectar recursos
3. **Servidores MCP**: Como criar com stdio
4. **MultiServerMCPClient**: Como conectar múltiplos servidores
5. **Comunicação stdio**: Protocolo local via stdin/stdout

## 🔑 Conceitos:

### MCP vs Tools:
- **Tools diretas**: Funções Python no código
- **MCP**: Protocolo externo
  - ✅ Reutilizável
  - ✅ Pode ser remoto
  - ✅ Padronizado

### Transportes MCP:
- **stdio**: Local (usado aqui)
- **sse**: Remoto via Server-Sent Events
- **http**: Requisições HTTP

## 🚀 Próximos Passos:

1. Crie servidores MCP para:
   - Banco de dados
   - APIs externas
   - Arquivos locais

2. Explore MCP públicos:
   - GitHub
   - Google Drive
   - Notion

3. Combine múltiplas fontes

## 📚 Recursos:

- [MCP Docs](https://modelcontextprotocol.io/)
- [LangChain MCP](https://github.com/langchain-ai/langchain-mcp)
- [Gemini API](https://ai.google.dev/)

---

**100% Gratuito** | Gemini + MCP